Jon Ball <br>
4/13/2022

# Regex : r'absen|(hasn\'t |not )(been |be )?attend|\bmissing|\bmissed'
<br>
This regex captures the meaning of absence, or missed class, or missing work. These are null outcomes that can also be counted (e.g., 4 missing assignments, 5 days absent). Teachers frequently mark down absences and missing assignments, which form the core of the grading process.
<br>
<br>
The regex is more precise when applied only to teacher rather than contact messages, because contacts tend to send more messages that contain "missed" or "missing" but are unrelated to academic outcomes.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
tp1 = pd.read_csv('data/rand100k_1.csv')
tp1 = tp1[tp1['TEXTENGLISH'].notnull()]
tp1.info()

## Objective: identify messages about students being absent or missing work with a simple regular expression

#### Let's create a regular expression for messages with the meaning of being absent (i.e., attendance), late, or missing (i.e., assignments).
<br>
It's easiest to build it from separate expressions.
<br>
<br>
First, let's make a helper function to print reproducible examples in batches of 30:

In [ ]:
def sample(df, regex):
    samp = df[df['TEXTENGLISH'].str.contains(regex, regex=True, case=False)].sample(n=30, random_state=30)
    for i, message in enumerate(samp['TEXTENGLISH'].tolist()):
        print('{}.'.format(i+1))
        print(message)
        print('\n--')

Absence:

In [ ]:
sample(tp1, r'absen')

All 30 of the messages above are about a student's absence or a notification/excuse for a student's absence.
<br>
This regex is not sensitive to whether a given absence is excused or necessary. It just identifies absences.
<br>
<br>

Let's broaden the regex so it captures other phrases like 'did not attend' or 'have not been attending' or 'will not be attending' that are synonymous with 'absent':

In [ ]:
sample(tp1, r'(hasn\'t |not )(been |be )?attend')

30/30 of the messages indicate a student absence. The regex captures relevant messages that 'absen' fails to. 
<br>
<br>
(However, teachers occasionally send scheduling emails with "please notify me if you cannot attend," which the regex also captures. Those aren't student absences. But it seems to be an infrequent false positive.)
<br>
<br>
Putting them both together:

In [ ]:
sample(tp1, r'absen|(hasn\'t |not )(been |be )?attend')

30/30 indicate a student absence, in the formal sense that a student is marked absent. The absence regex is precise!

#### Finally, let's try a regex that captures 'missing', as in 'missing work' or 'missing class':

In [ ]:
#The word boundary character '\b' prevents the regex from capturing 'dismissing'
sample(tp1, r'\bmissing')

All 30 of the messages above indicate a missing assignment, missing work, or missing class. These would be marked by a teacher as a 0 / missing / absent. <b>The regex captures academic outcomes.</b>
<br>
(And perhaps the occasional message from a parent looking for a missing item.)
<br>
<br>
Let's try the past tense:

In [ ]:
#The word boundary character '\b' prevents the regex from capturing 'dismissed'
sample(tp1, r'\bmissed')

'missed' is fairly precise. But some messages are about "missed breakfast" or "missed the bus" or missed contact-teacher calls. The simple solution would be to run the regex only over messages sent by teachers.

In [ ]:
sample(tp1, r'\bmissed|\bmissing')

Expressions such as "if you missed it" or "missed out on" or "you will be missed" imply absence. But a small fraction of the messages captured by '\bmissed' will be idiomatic and unrelated to academic outcomes.

In [ ]:
sample(tp1, r'\bmisses')

Present tense 'misses' is imprecise. The present tense regex captures messages that a student misses their teacher and peers, or that a student misses directions often. The past and present participle of 'miss' are more precise:
<br>
<br>
r'\bmissing|\bmissed'